# Problem 3: Risk-Minimizing NIPT Scheduling

Optimize one-test and two-test schedules across six BMI groups while accounting for assay variation and model uncertainty.

## Data and reproducibility

The participant-level NIPT dataset is not distributed in this public repository. To reproduce the analysis, place an authorized copy at `data/nipt_data.xlsx` using the English schema documented in `data/README.md`.


## Setup


In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_PATH = REPO_ROOT / "data" / "nipt_data.xlsx"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The authorized NIPT dataset is not included in this public repository. "
        "Place it at data/nipt_data.xlsx after reviewing data-use restrictions."
    )


## Risk model and optimized schedules


In [ ]:


import os, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.tree import DecisionTreeRegressor


FILE_PATH = DATA_PATH
THR_RAW = 0.04
CV_BASE = 0.10
Z_LOW   = 1.645
RANDOM_STATE = 42


STEP = 1/7.0
WEEKS_FULL = np.round(np.arange(9.0, 32.0 + STEP/2, STEP), 6)

WEEKS_OPT  = WEEKS_FULL[(WEEKS_FULL >= 10.0) & (WEEKS_FULL <= 20.0)]
TARGET_COVER = 0.80


RISK_WEIGHT = {"early":1.0, "mid":3.0, "late":8.0}
EARLY_CUT, LATE_CUT = 12.0, 28.0
W_FAIL = RISK_WEIGHT["late"]*2.0


def set_chinese_font():
    candidates = [
        r"", r"",
        r"",
        "/System/Library/Fonts/PingFang.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJKsc-Regular.otf",
    ]
    for fp in candidates:
        if os.path.exists(fp):
            try:
                font_manager.fontManager.addfont(fp)
                font_name = font_manager.FontProperties(fname=fp).get_name()
                matplotlib.rcParams["font.sans-serif"] = [font_name, "DejaVu Sans", "Arial"]
                break
            except Exception:
                pass
    matplotlib.rcParams["axes.unicode_minus"] = False
set_chinese_font()


def _find_col(cols, keys):
    cols_ns = [str(c).strip().replace(" ", "") for c in cols]
    for k in keys:
        k = str(k).strip().replace(" ", "")
        for i, c in enumerate(cols_ns):
            if k in c:
                return cols[i]
    return None

def _parse_week(x):

    if pd.isna(x): return np.nan
    s = str(x).strip()
    m = re.match(r"^\s*(\d+)\s*\+\s*(\d+)\s*/\s*7\s*$", s)
    if m: return int(m.group(1)) + int(m.group(2))/7.0
    m = re.findall(r"(\d+)\s*(?:week|w|W)?\s*\+?\s*(\d+)?", s)
    if m:
        w = int(m[0][0]); d = int(m[0][1]) if m[0][1] not in [None, ""] else 0
        if "+" in s or "week" in s or "w" in s.lower(): return w + d/7.0
    try: return float(s)
    except: return np.nan

def fmt_week_7(x):

    if not np.isfinite(x): return ""
    w = int(np.floor(x + 1e-9))
    d = int(round((x - w) * 7))
    if d == 0: return f"{w}"
    if d == 7: return f"{w+1}"
    return f"{w}+{d}/7"

def time_weight(t):
    if t <= EARLY_CUT: return RISK_WEIGHT["early"]
    if t <  LATE_CUT:  return RISK_WEIGHT["mid"]
    return RISK_WEIGHT["late"]

def expected_risk_once(Pt, weeks):
    W = np.array([time_weight(t) for t in weeks])
    return Pt*W + (1-Pt)*W_FAIL

def expected_risk_twice(Pt, weeks):
    W = np.array([time_weight(t) for t in weeks])
    best = (np.inf, None, None)
    for i in range(len(weeks)):
        for j in range(i+1, len(weeks)):
            P1, P2 = Pt[i], Pt[j]
            risk = P1*W[i] + (1-P1)*(P2*W[j] + (1-P2)*W_FAIL)
            if risk < best[0]:
                best = (risk, weeks[i], weeks[j])
    return best


raw = pd.read_excel(FILE_PATH, sheet_name=0)
cols = list(raw.columns)

col_age   = _find_col(cols, ["maternal_age","maternal_age","C"])
col_ht    = _find_col(cols, ["maternal_height_cm","maternal_height_cm","D"])
col_wt    = _find_col(cols, ["maternal_weight_kg","maternal_weight_kg","E"])
col_ga    = _find_col(cols, ["gestational_age","gestational_age","J"])
col_bmi   = _find_col(cols, ["maternal_bmi","BMI","K"])
col_yconc = _find_col(cols, ["y_chromosome_fraction","y_chromosome_fraction","V"])
col_gc    = _find_col(cols, ["gc_content","gc_content","P"])
col_L     = _find_col(cols, ["raw_read_count","raw_read_count","L"])
col_M     = _find_col(cols, ["mapping_ratio","mapping_ratio","M"])
col_N     = _find_col(cols, ["duplicate_ratio","N"])
col_O     = _find_col(cols, ["unique_read_count","O"])
col_AA    = _find_col(cols, ["filtered_read_ratio","AA"])

need = [col_ga, col_bmi, col_yconc, col_gc]
assert all(c is not None for c in need), "Missing required columns: gestational age, BMI, Y-chromosome fraction, or GC content"

df = raw.copy()
df[col_ga] = df[col_ga].apply(_parse_week)
for c in [col_bmi, col_age, col_ht, col_wt, col_yconc, col_gc, col_L, col_M, col_N, col_O, col_AA]:
    if c is not None:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[(df[col_gc] >= 0.40) & (df[col_gc] <= 0.60)].dropna(subset=[col_ga, col_bmi, col_yconc]).reset_index(drop=True)


feature_cols = [c for c in [col_ga, col_bmi, col_age, col_ht, col_wt, col_L, col_M, col_N, col_O, col_AA, col_gc] if c]
for c in feature_cols:
    df[c] = df[c].fillna(df[c].median())


if col_bmi in df: df["BMI2"] = df[col_bmi]**2
if col_age in df: df["AGE2"] = df[col_age]**2
if col_ga  in df: df["GA2"]  = df[col_ga]**2
feature_cols_ext = feature_cols + [c for c in ["BMI2","AGE2","GA2"] if c in df.columns]

X = df[feature_cols_ext].astype(float).values
y = df[col_yconc].astype(float).values


kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_pred = np.zeros_like(y, dtype=float)
for tr, va in kf.split(X):
    reg = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=4)
    reg.fit(X[tr], y[tr])
    oof_pred[va] = reg.predict(X[va])
resid = y - oof_pred
mad = np.median(np.abs(resid - np.median(resid)))
sigma = 1.4826*mad if np.isfinite(mad) and mad > 0 else np.std(resid, ddof=1)


reg_full = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=4).fit(X, y)
N, T = len(df), len(WEEKS_FULL)
X_blocks = []
for t in WEEKS_FULL:
    Xt = df[feature_cols_ext].copy()
    Xt[col_ga] = t
    if "GA2" in Xt.columns:
        Xt["GA2"] = t**2
    X_blocks.append(Xt.values)
mu_all = reg_full.predict(np.vstack(X_blocks)).reshape(T, N).T
mu_all = np.maximum.accumulate(mu_all, axis=1)


thr = THR_RAW * (1 + CV_BASE)
meet = mu_all - Z_LOW * sigma >= thr
g_star = np.full(N, np.nan, dtype=float)
for i in range(N):
    m = meet[i]
    if m.any():
        g_star[i] = WEEKS_FULL[np.argmax(m)]
df["g_star"] = g_star


risk = np.full(N, 1, dtype=int)
risk[g_star <= 12.0] = 0
risk[g_star >= 28.0] = 2
df["risk_level"] = risk


bmi = df[col_bmi].values
valid = np.isfinite(g_star)
edges = None
if valid.sum() >= 120:
    rtree = DecisionTreeRegressor(max_leaf_nodes=6, min_samples_leaf=60, random_state=RANDOM_STATE)
    rtree.fit(bmi[valid].reshape(-1, 1), g_star[valid])
    thr_list = []
    def _walk(n):
        L, R = rtree.tree_.children_left[n], rtree.tree_.children_right[n]
        if L != -1 and R != -1:
            thr_list.append(rtree.tree_.threshold[n])
            _walk(L); _walk(R)
    _walk(0)
    thr_list = sorted([t for t in thr_list if np.isfinite(t)])
    bmin, bmax = float(bmi.min()), float(bmi.max())
    edges = [bmin - 1e-6] + [t for t in thr_list if bmin < t < bmax] + [bmax + 1e-6]
if edges is None or len(edges) < 7:
    qs = np.unique(np.quantile(bmi, np.linspace(0, 1, 7)))
    if len(qs) < 7:
        qs = np.linspace(bmi.min(), bmi.max(), 7)
    edges = list(qs)
edges = sorted(list(edges))
if len(edges) > 7:
    edges = [edges[0]] + list(np.linspace(edges[1], edges[-2], 5)) + [edges[-1]]
while len(edges) < 7:
    gaps = np.diff(edges)
    k = int(np.argmax(gaps))
    edges.insert(k+1, (edges[k] + edges[k+1]) / 2)

BMI_LABELS = [f"{edges[i]:.1f}–{edges[i+1]:.1f}" for i in range(6)]
df["bmi_group"] = pd.cut(df[col_bmi], bins=edges, right=False, labels=BMI_LABELS, include_lowest=True)


def group_cover_curve(g_star_vec):

    return np.array([np.nanmean(g_star_vec <= t) for t in WEEKS_FULL])

rows = []
for lab in BMI_LABELS:
    idx = np.where(df["bmi_group"].astype(str).values == lab)[0]
    if idx.size == 0:
        rows.append({"bmi_group": lab, "sample_size": 0,
                     f"coverage_at_least_{int(TARGET_COVER*100)}%earliest_week": "",
                     "optimal_week_one_test": "", "optimal_week_two_tests_t1": "", "optimal_week_two_tests_t2": ""})
        continue

    cov_full = group_cover_curve(df.loc[idx, "g_star"].values)
    mask = (WEEKS_FULL >= 10.0) & (WEEKS_FULL <= 20.0)
    weeks = WEEKS_FULL[mask]; cov = cov_full[mask]


    hit = np.where(cov >= TARGET_COVER)[0]
    t_cov = weeks[hit[0]] if hit.size else np.nan

    R1 = expected_risk_once(cov, weeks)
    i1 = int(np.nanargmin(R1)); t1 = weeks[i1]
    _, t1x, t2x = expected_risk_twice(cov, weeks)

    rows.append({
        "bmi_group": lab, "sample_size": int(idx.size),
        f"coverage_at_least_{int(TARGET_COVER*100)}%earliest_week": fmt_week_7(float(t_cov)) if np.isfinite(t_cov) else "",
        "optimal_week_one_test": fmt_week_7(float(t1)),
        "optimal_week_two_tests_t1": fmt_week_7(float(t1x)),
        "optimal_week_two_tests_t2": fmt_week_7(float(t2x)),
    })

summary = pd.DataFrame(rows)


grp = (df.groupby(["bmi_group", "risk_level"], observed=True)
         .size()
         .unstack(fill_value=0)
         .reindex(BMI_LABELS)
         .fillna(0))

x = np.arange(len(BMI_LABELS))
plt.figure(figsize=(8.4, 5.2))
plt.bar(x, grp.get(0, 0).values, label="Low risk (0)")
plt.bar(x, grp.get(1, 0).values, bottom=grp.get(0, 0).values, label="Moderate risk (1)")
plt.bar(x, grp.get(2, 0).values, bottom=(grp.get(0, 0).values + grp.get(1, 0).values), label="high_risk（2）")
plt.xticks(x, BMI_LABELS, rotation=20)
plt.xlabel("BMI group"); plt.ylabel("Count")
plt.title("Risk distribution across six BMI groups")
plt.legend()
plt.tight_layout()


out_dir = str(OUTPUT_DIR)
png_path  = os.path.join(out_dir, "problem-3-bmi-risk-distribution.png")
xlsx_path = os.path.join(out_dir, "problem-3-strategies-six-groups.xlsx")

plt.savefig(png_path, dpi=180)
with pd.ExcelWriter(xlsx_path) as xw:
    summary.to_excel(xw, sheet_name="group_strategy", index=False)
    pd.DataFrame({
        "bmi_group_label": BMI_LABELS,
        "lower_bound": edges[:-1],
        "upper_bound": edges[1:]
    }).to_excel(xw, sheet_name="bmi_group_boundaries", index=False)

print("Figure saved：", png_path)
print("Table saved：", xlsx_path)
print("Six BMI-group boundaries：", [round(e, 2) for e in edges])


## Sensitivity analysis


In [ ]:
#Error sensitivity heatmap（CV × α → t_{k,p0}) 
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

def setup_plot_font(prefer=("Microsoft YaHei", "SimHei", "SimSun",
                          "Noto Sans CJK SC", "Source Han Sans CN")):
    installed = {f.name for f in fm.fontManager.ttflist}
    for fam in prefer:
        if fam in installed:
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            return
    for p in [r"", r"", r""]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            fam = fm.FontProperties(fname=p).get_name()
            matplotlib.rcParams["font.family"] = "sans-serif"
            matplotlib.rcParams["font.sans-serif"] = [fam]
            matplotlib.rcParams["axes.unicode_minus"] = False
            matplotlib.rcParams["pdf.fonttype"] = 42
            matplotlib.rcParams["ps.fonttype"]  = 42
            return
setup_plot_font()


need_build = False
for name in ["df","label_order","col_bmi","weeks","gstar_array","coverage_curve"]:
    if name not in globals():
        need_build = True
        break

if need_build:

    def _safe_read_excel(candidates=None, sheet_candidates=None, engine="openpyxl"):
        cand = (candidates or []) + [
            r"nipt_data.xlsx",
        ]
        cand += glob.glob("**/nipt_data.xlsx", recursive=True)
        fpath = None
        for p in cand:
            if p and os.path.exists(p):
                fpath = p; break
        xls = pd.ExcelFile(fpath, engine=engine)
        sheet = xls.sheet_names[0]
        print(f"[OK] Loaded: {os.path.abspath(fpath)} | Sheet: {sheet}")
        return pd.read_excel(fpath, sheet_name=sheet)

    def _find_col(cols, keys):
        cols = [str(c).strip() for c in cols]
        for k in keys:
            for c in cols:
                if k in c: return c
        return None

    def _parse_week(x):
        if pd.isna(x): return np.nan
        s = str(x).strip()
        m = re.findall(r"(\d+)\s*(?:w|week)\s*\+?\s*(\d+)?", s, flags=re.I)
        if m:
            w = int(m[0][0]); d = int(m[0][1]) if m[0][1] not in [None,""] else 0
            return w + d/7.0
        try: return float(s)
        except: return np.nan

    raw = _safe_read_excel()
    col_bmi   = _find_col(raw.columns, ["maternal_bmi","BMI"])
    col_ga    = _find_col(raw.columns, ["gestational_age","gestational_age"])
    col_yconc = _find_col(raw.columns, ["y_chromosome_fraction","y_chromosome_fraction","y_chromosome_fraction"])
    df = raw[[col_bmi, col_ga, col_yconc]].copy()
    df[col_bmi]   = pd.to_numeric(df[col_bmi], errors="coerce")
    df[col_ga]    = df[col_ga].apply(_parse_week)
    df[col_yconc] = pd.to_numeric(df[col_yconc], errors="coerce")
    df = df.dropna().reset_index(drop=True)


    weeks = np.array([10 + i/7 for i in range(0, 11*7)])
    P0 = 0.80


    try:
        from xgboost import XGBRegressor
        reg = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                           subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                           objective="reg:squarederror", tree_method="hist",
                           random_state=42, n_jobs=1
                           ).fit(df[[col_ga,col_bmi]].astype(float), df[col_yconc].astype(float))
    except Exception:
        from sklearn.ensemble import RandomForestRegressor
        reg = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=1
                                    ).fit(df[[col_ga,col_bmi]].astype(float), df[col_yconc].astype(float))
    yhat  = reg.predict(df[[col_ga,col_bmi]])
    sigma = float(np.std(df[col_yconc] - yhat, ddof=1))
    Z095  = 1.645

    # BMI group
    bins   = [24, 28, 30, 32, 34, 36, np.inf]
    labels = ["24–28","28–30","30–32","32–34","34–36","≥36"]
    df["bmi_group"] = pd.cut(df[col_bmi], bins=bins, labels=labels, right=False)
    label_order = [lab for lab in labels if lab in df["bmi_group"].dropna().unique()]


    def gstar_array(bmis, cv=0.10, sigma_scale=1.0):
        bmis = np.asarray(bmis, dtype=float)
        T = len(weeks); M = len(bmis)
        GA  = np.tile(weeks[:,None], (1,M))
        BMI = np.tile(bmis[None,:], (T,1))
        Xp  = pd.DataFrame({col_ga: GA.ravel(), col_bmi: BMI.ravel()})
        mu  = reg.predict(Xp).reshape(T,M)
        L   = mu - Z095 * sigma * sigma_scale
        tau = 0.04 * (1 + cv)
        mask = L >= tau
        ok   = mask.any(axis=0)
        first_idx = np.where(ok, mask.argmax(axis=0), -1)
        gstars = np.where(ok, weeks[first_idx], np.nan)
        return gstars

    def coverage_curve(gs):
        return np.array([float(np.nanmean(gs <= t)) for t in weeks])


CV_GRID     = [0.05, 0.10, 0.15]
ALPHA_GRID  = [0.8,  1.0,  1.2]
P0          = globals().get("P0", 0.80)

def _t_cov_for_group(sub_df, cv, alpha):
    bmis = sub_df[col_bmi].values
    gs   = gstar_array(bmis, cv=cv, sigma_scale=alpha)
    cov  = coverage_curve(gs)
    idx  = np.where(cov >= P0)[0]
    if len(idx)==0:

        t_cov = min(20.0, weeks[-1])
    else:
        t_cov = min(20.0, weeks[idx[0]])
    return float(t_cov)


mats = {}
glob_min, glob_max =  1e9, -1e9
for g in label_order:
    sub = df[df["bmi_group"]==g]
    if len(sub) < 5: 
        continue
    M = np.zeros((len(ALPHA_GRID), len(CV_GRID)), dtype=float)
    for i, alpha in enumerate(ALPHA_GRID):
        for j, cv in enumerate(CV_GRID):
            M[i, j] = _t_cov_for_group(sub, cv=cv, alpha=alpha)
    mats[g] = M
    glob_min = min(glob_min, np.nanmin(M))
    glob_max = max(glob_max, np.nanmax(M))


n = len(mats)
ncols = min(3, n if n>0 else 1)
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.5*nrows), constrained_layout=True)
if nrows*ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = np.array([axes])

last_im = None
k = 0
for g, M in mats.items():
    r, c = divmod(k, ncols); k += 1
    ax = axes[r, c]
    last_im = ax.imshow(M, vmin=glob_min, vmax=glob_max, origin="upper")

    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i,j]:.1f}", ha="center", va="center", color="w", fontsize=10)
    ax.set_xticks(range(len(CV_GRID)));     ax.set_xticklabels([f"{int(100*c)}%" for c in CV_GRID])
    ax.set_yticks(range(len(ALPHA_GRID)));  ax.set_yticklabels([f"×{a:.1f}" for a in ALPHA_GRID])
    ax.set_xlabel("Assay coefficient of variation (CV)")
    ax.set_ylabel("Residual scale factor alpha")
    ax.set_title(f"{g}：First week reaching at least {int(P0*100)}% coverage; gestational week t$_{{k,p0}}$")


for idx in range(k, nrows*ncols):
    r, c = divmod(idx, ncols)
    axes[r, c].axis("off")


if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes, shrink=0.92)
    cbar.set_label("gestational_age（week）")

fig.suptitle("Error sensitivity heatmap: effect of CV and alpha on the first week reaching target coverage t$_{k,p0}$ ", y=1.02, fontsize=13)
plt.show()
